In [15]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm
from matplotlib.patches import Rectangle
import seaborn as sns

def is_float(x):
    try:
        float(x)
        return True
    except ValueError:
        return False

def parse_abd_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    blocks = content.split('Generating Sheet ')[1:]
    results = []

    for block in blocks:
        lines = block.strip().split('\n')
        name = lines[0].split(' (1x1x1)')[0].strip()

        mat_start = -1
        for i, line in enumerate(lines):
            if '--- [ABD] Full Matrix (6x6) ---' in line:
                mat_start = i + 1
                break

        if mat_start != -1:
            mat_str = " ".join(lines[mat_start:])
            mat_str = mat_str.replace('[', ' ').replace(']', ' ')
            vals = [float(x) for x in mat_str.split() if is_float(x)]

            if len(vals) >= 36:
                mat = np.array(vals[:36]).reshape((6, 6))
                results.append((name, mat))

    return results

def main():
    # ================= 字体与全局设置 =================
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Times New Roman']
    plt.rcParams['font.size'] = 9
    plt.rcParams['mathtext.fontset'] = 'stix'

    file_3d = r'E:\~Paper\2026_ThinWalledHomo\5_Figures\Fig3.1\Sheet TPMS 3D.txt'
    file_2d = r'E:\~Paper\2026_ThinWalledHomo\5_Figures\Fig3.1\Sheet TPMS 2D.txt'

    if not os.path.exists(file_3d) or not os.path.exists(file_2d):
        print(f"找不到文件，请检查路径是否正确！")
        return

    data_3d = parse_abd_file(file_3d)
    data_2d = parse_abd_file(file_2d)
    num_structs = min(len(data_3d), len(data_2d))

    # ================= 莫兰迪高级单向色系配置 =================
    # 【修改点】刚度色图(主图): 奶白 -> 莫兰迪粉 -> 莫兰迪深红 (提取了红色的那一半)
    cmap_val = sns.blend_palette(['#F3F4F6', '#DDA7A5', '#9B5D5E'], as_cmap=True)

    # 误差色图: 奶白 -> 莫兰迪灰绿 (对比色，区分误差与绝对数值)
    cmap_err = sns.blend_palette(['#F3F4F6', '#B5C2B7', '#6A8074'], as_cmap=True)

    for i in range(num_structs):
        name_3d, mat_3d = data_3d[i]
        name_2d, mat_2d = data_2d[i]
        struct_name = name_2d.replace('Sheet ', '')

        # 将极小负数截断误差归零
        mat_3d[mat_3d < 0] = 0
        mat_2d[mat_2d < 0] = 0

        # 计算相对误差
        err_mat = np.zeros_like(mat_2d)
        valid_mask = mat_2d > 1.0
        err_mat[valid_mask] = np.abs((mat_3d[valid_mask] - mat_2d[valid_mask]) / mat_2d[valid_mask]) * 100.0

        # 强制锁死 0 ~ 10000 范围，使用对数拉伸中间数值
        norm_val = SymLogNorm(linthresh=10, linscale=1, vmin=0, vmax=10000, base=10)

        # ================= 创建 12cm x 3cm 画布 =================
        fig, axes = plt.subplots(1, 3, figsize=(12/2.54, 3/2.54))
        plt.subplots_adjust(left=0.02, right=0.95, top=0.95, bottom=0.05, wspace=0.35)

        im_3d = axes[0].imshow(mat_3d, cmap=cmap_val, norm=norm_val)
        im_2d = axes[1].imshow(mat_2d, cmap=cmap_val, norm=norm_val)
        im_err = axes[2].imshow(err_mat, cmap=cmap_err, vmin=0, vmax=max(10, np.max(err_mat)))

        for ax in axes:
            ax.set_xticks([])
            ax.set_yticks([])

            # A, B, D 矩阵实线框
            ax.add_patch(Rectangle((-0.5, -0.5), 3, 3, fill=False, edgecolor='black', lw=0.8, ls='-'))
            ax.text(-0.3, -0.3, 'A', ha='left', va='top', fontsize=9, fontweight='bold', color='black')

            ax.add_patch(Rectangle((2.5, -0.5), 3, 3, fill=False, edgecolor='black', lw=0.8, ls='-'))
            ax.text(2.7, -0.3, 'B', ha='left', va='top', fontsize=9, fontweight='bold', color='black')
            ax.add_patch(Rectangle((-0.5, 2.5), 3, 3, fill=False, edgecolor='black', lw=0.8, ls='-'))
            ax.text(-0.3, 2.7, 'B', ha='left', va='top', fontsize=9, fontweight='bold', color='black')

            ax.add_patch(Rectangle((2.5, 2.5), 3, 3, fill=False, edgecolor='black', lw=0.8, ls='-'))
            ax.text(2.7, 2.7, 'D', ha='left', va='top', fontsize=9, fontweight='bold', color='black')

        # ================= 自定义加宽单向 Colorbar =================
        custom_ticks = [0, 100, 10000]
        custom_ticklabels = ['$0$', '$10^2$', '$10^4$']

        # 1. 3D ABD Colorbar
        cbar0 = fig.colorbar(im_3d, ax=axes[0], shrink=0.85, aspect=12, pad=0.04, ticks=custom_ticks)
        cbar0.ax.set_yticklabels(custom_ticklabels, fontsize=8)

        # 2. 2D ABD Colorbar
        cbar1 = fig.colorbar(im_2d, ax=axes[1], shrink=0.85, aspect=12, pad=0.04, ticks=custom_ticks)
        cbar1.ax.set_yticklabels(custom_ticklabels, fontsize=8)

        # 3. 误差 Error Colorbar
        cbar2 = fig.colorbar(im_err, ax=axes[2], shrink=0.85, aspect=12, pad=0.04)
        cbar2.ax.tick_params(labelsize=8)
        cbar2.set_label('%', rotation=0, labelpad=2, fontsize=8, y=1.05)

        output_filename = f"ABD_Heatmap_{struct_name}.png"
        plt.savefig(output_filename, dpi=400, bbox_inches='tight')
        plt.close()

        print(f"已生成【莫兰迪红】主色调图片: {output_filename}")

if __name__ == "__main__":
    main()

已生成【莫兰迪红】主色调图片: ABD_Heatmap_Primitive.png
已生成【莫兰迪红】主色调图片: ABD_Heatmap_Diamond.png
已生成【莫兰迪红】主色调图片: ABD_Heatmap_Gyroid.png
已生成【莫兰迪红】主色调图片: ABD_Heatmap_I-WP.png
已生成【莫兰迪红】主色调图片: ABD_Heatmap_F-RD.png
已生成【莫兰迪红】主色调图片: ABD_Heatmap_L.png
已生成【莫兰迪红】主色调图片: ABD_Heatmap_Tubular P.png
已生成【莫兰迪红】主色调图片: ABD_Heatmap_Tubular G.png
已生成【莫兰迪红】主色调图片: ABD_Heatmap_I2-Y.png


In [18]:
import os
import numpy as np

def is_float(x):
    try:
        float(x)
        return True
    except ValueError:
        return False

def parse_abd_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    blocks = content.split('Generating Sheet ')[1:]
    results = [] # 改用 List，保留文件的先后顺序

    for block in blocks:
        lines = block.strip().split('\n')
        name = lines[0].split(' (1x1x1)')[0].strip()

        mat_start = -1
        for i, line in enumerate(lines):
            if '--- [ABD] Full Matrix (6x6) ---' in line:
                mat_start = i + 1
                break

        if mat_start != -1:
            mat_str = " ".join(lines[mat_start:])
            mat_str = mat_str.replace('[', ' ').replace(']', ' ')
            vals = [float(x) for x in mat_str.split() if is_float(x)]

            if len(vals) >= 36:
                mat = np.array(vals[:36]).reshape((6, 6))
                mat[mat < 0] = 0 # 剔除极小负数截断误差
                results.append((name, mat))

    return results

def main():
    file_3d = r'E:\~Paper\2026_ThinWalledHomo\5_Figures\Fig3.1\Sheet TPMS 3D.txt'
    file_2d = r'E:\~Paper\2026_ThinWalledHomo\5_Figures\Fig3.1\Sheet TPMS 2D.txt'

    if not os.path.exists(file_3d) or not os.path.exists(file_2d):
        print(f"找不到文件，请检查路径是否正确！")
        return

    # 这里解析出来的都是按文件里原本顺序排列的 List
    data_3d = parse_abd_file(file_3d)
    data_2d = parse_abd_file(file_2d)

    num_structs = min(len(data_3d), len(data_2d))

    # 目标结构及展示名称字典
    target_structs = {
        'Gyroid': 'G (Gyroid)',
        'Diamond': 'D (Diamond)',
        'Primitive': 'P (Primitive)',
        'I-WP': 'W (I-WP)'
    }

    # 需要提取对比的 6 个核心刚度分量
    components = [
        ("A11 (In-plane Tension)", 0, 0),
        ("A22 (In-plane Tension)", 1, 1),
        ("A66 (In-plane Shear)",   2, 2),
        ("D11 (Out-plane Bending)", 3, 3),
        ("D22 (Out-plane Bending)", 4, 4),
        ("D66 (Out-plane Twist)",   5, 5),
    ]

    print("="*85)
    print(f"{'Structure':<15} | {'Component':<25} | {'3D-VH (Eq)':>10} | {'2D-PH (Ours)':>12} | {'Error (%)':>10}")
    print("="*85)

    # 按照顺序（Index）一一配对
    for i in range(num_structs):
        name_3d, mat_3d = data_3d[i]
        name_2d, mat_2d = data_2d[i]

        # 统一以 2D 文件中规范的名字（例如 "Primitive"）作为判定基准
        struct_key = name_2d.replace('Sheet ', '').strip()

        # 如果当前按顺序读到的结构属于 G, D, P, W 的其中一个，就输出分析
        if struct_key in target_structs:
            display_name = target_structs[struct_key]

            print(f"\n[{display_name}]  <-- matched: 3D({name_3d}) vs 2D({name_2d})")
            print("-" * 85)

            for comp_name, row, col in components:
                val_3d = mat_3d[row, col]
                val_2d = mat_2d[row, col]

                if val_2d > 1e-3:
                    error = abs((val_3d - val_2d) / val_2d) * 100.0
                    error_str = f"{error:>8.2f} %"
                else:
                    error_str = "   N/A    "

                print(f"{' ' * 15} | {comp_name:<25} | {val_3d:>10.2f} | {val_2d:>12.2f} | {error_str}")

    print("\n" + "="*85)
    print("* Note: Error = |(3D_VH - 2D_PH) / 2D_PH| * 100%")

if __name__ == "__main__":
    main()

Structure       | Component                 | 3D-VH (Eq) | 2D-PH (Ours) |  Error (%)

[P (Primitive)]  <-- matched: 3D(Sheet G) vs 2D(Primitive)
-------------------------------------------------------------------------------------
                | A11 (In-plane Tension)    |     674.00 |       357.44 |    88.56 %
                | A22 (In-plane Tension)    |     674.00 |       357.44 |    88.56 %
                | A66 (In-plane Shear)      |     245.31 |       309.70 |    20.79 %
                | D11 (Out-plane Bending)   |    5616.63 |      2236.89 |   151.09 %
                | D22 (Out-plane Bending)   |    5616.63 |      2236.89 |   151.09 %
                | D66 (Out-plane Twist)     |    2044.26 |      2030.32 |     0.69 %

[D (Diamond)]  <-- matched: 3D(Sheet D) vs 2D(Diamond)
-------------------------------------------------------------------------------------
                | A11 (In-plane Tension)    |     787.51 |       756.68 |     4.07 %
                | A22 (In-plane 